In [0]:
from pyspark.sql import functions as F
CATALOG_SCHEMA = "workspace.stocks"
FACT = f"{CATALOG_SCHEMA}.fact_daily_price"     
LOOKBACK_DAYS = 3                               

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FACT} (
  ticker STRING, trade_date DATE,
  open DECIMAL(18,4), high DECIMAL(18,4), low DECIMAL(18,4), close DECIMAL(18,4),
  volume BIGINT, loaded_at TIMESTAMP
) USING DELTA
""")

DataFrame[]

In [0]:
watermark = spark.table(FACT).agg(F.max("trade_date")).collect()[0][0]
print("bookmark:", watermark)       

bookmark: 2026-10-28


In [0]:
silver = spark.table(f"{CATALOG_SCHEMA}.silver_prices")

if watermark is None:
    pile = silver                                        
else:
    start = F.date_sub(F.lit(watermark), LOOKBACK_DAYS)         
    pile = silver.filter(F.col("trade_date") > start) 

pile = pile.select("ticker", "trade_date", "open", "high", "low", "close", "volume")
pile.createOrReplaceTempView("new_prices")
print("lines in today's pile:", pile.count())

lines in today's pile: 24


In [0]:
display(spark.sql(f"""
MERGE INTO {FACT} AS t
USING new_prices AS s
ON t.ticker = s.ticker AND t.trade_date = s.trade_date              
WHEN MATCHED AND (                                                
       NOT (t.close  <=> s.close)  OR NOT (t.open <=> s.open)
    OR NOT (t.high   <=> s.high)   OR NOT (t.low  <=> s.low)
    OR NOT (t.volume <=> s.volume)
) THEN UPDATE SET
       t.open = s.open, t.high = s.high, t.low = s.low,
       t.close = s.close, t.volume = s.volume, t.loaded_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT                                       
       (ticker, trade_date, open, high, low, close, volume, loaded_at)
       VALUES (s.ticker, s.trade_date, s.open, s.high, s.low, s.close, s.volume, current_timestamp())
"""))


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
f = spark.table(FACT)
print("lines in book:", f.count())
print("unique (ticker, day):", f.select("ticker", "trade_date").distinct().count())
print("latest day:", f.agg(F.max("trade_date")).collect()[0][0])

lines in book: 808
unique (ticker, day): 808
latest day: 2026-10-28
